# Telco Data Churn - Prediction (Model Creation and analysis)
---
-- **Running notebook ``Exploratory Data Analysis\EDA.ipynb`` is required in order to retrieve the cleaned and engineered data CSVs which are used in this notebook.** --
## Specific Aim: Prove correct analysis of data by prediction
We extend on the data analysis of this project by using machine learning to attempt to make predictions given the data cleaning and engineering. \
A secondary aim in using machine learning modes is to make clear the difference in model efficiency when given cleaned vs engineered data. As well as analyse what a "good" classifier for this data would be, as there are several many choices.

In [ ]:
# Imports
import pandas as pd
import numpy as np

# Read data for use
dfClean = pd.read_csv('../data/Telco_Cleaned.csv')
dfEng = pd.read_csv('../data/Telco_Engineered.csv')

We then begin the machine learning pipeline, first split the data into train and test sets first:

In [9]:
from sklearn.model_selection import train_test_split

# Separate customer data from their 'Churn' status, teach the model that "given this information, a customer can churn or not"
XClean = dfClean.drop('Churn', axis = 1)           
yClean = dfClean['Churn']

XEng = dfEng.drop('Churn', axis = 1)           
yEng = dfEng['Churn']

# Then, split the data given into training data (what it learns from) and test data (what it attempts to predict)
XC_train, XC_test, yC_train, yC_test = train_test_split(XClean, yClean, test_size=0.2, random_state=42)
XE_train, XE_test, yE_train, yE_test = train_test_split(XEng, yEng, test_size=0.2, random_state=42)

Then, test the model using the different models:

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Logistic Regression: uses sigmoid to 'fit datapoints to a curve', classifying points based on how close they are to 0 or 1
# More rigorously, weights are learned for each feature, then the weighted sum of each feature is computed and passed through sigmoid. If the value of \sigma(customer sum) is greater than 0.5,
# then predict that they churned. Otherwise, predict that they did not churn. Quite steady right?
lrC = LogisticRegression(max_iter=1000)
lrC.fit(XC_train, yC_train)
lrC_preds = lrC.predict(XC_test)
print(classification_report(yC_test, lrC_preds))

              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1031
           1       0.62      0.54      0.58       376

    accuracy                           0.79      1407
   macro avg       0.73      0.71      0.72      1407
weighted avg       0.78      0.79      0.79      1407



We see that on the clean dataset, the logistic regression model predicts with no churn correctly with probability 0.84, whereas it predicts if customers actually churn correctly with probability 0.62. 

However, its low recall stands as an issue - it correctly predicted 54% of actually churning customers. If the aim is to target customers that are likely to churn and offer them solutions to cause them to remain in the company, then losing almost 50% of these numbers is nowhere near the numbers a model should aim to reach. 

In [12]:
# Now testing Logistic regression on engineered datasets:
lrE = LogisticRegression(max_iter=1000)
lrE.fit(XE_train, yE_train)
lrE_preds = lrE.predict(XE_test)
print(classification_report(yE_test, lrE_preds))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1031
           1       0.65      0.52      0.58       376

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.79      0.80      0.79      1407



The most interesting finding of all - The engineered data slightly improved recall for predicting 'no churn' as well as precision for 'churn', but decreased recall for 'churn'. The trade-off for engineering data seems to be quite small, at least in logistic regression. We now test random forest:

In [13]:
from sklearn.ensemble import RandomForestClassifier

# Clean data
rfC = RandomForestClassifier(random_state=42)
rfC.fit(XC_train, yC_train)
rfC_preds = rfC.predict(XC_test)
print(classification_report(yC_test, rfC_preds))

              precision    recall  f1-score   support

           0       0.82      0.89      0.85      1031
           1       0.61      0.47      0.53       376

    accuracy                           0.78      1407
   macro avg       0.71      0.68      0.69      1407
weighted avg       0.76      0.78      0.77      1407



In [15]:
# Engineered data
rfE = RandomForestClassifier(random_state=42)
rfE.fit(XE_train, yE_train)
rfE_preds = rfE.predict(XE_test)
print(classification_report(yE_test, rfE_preds))

              precision    recall  f1-score   support

           0       0.82      0.90      0.86      1031
           1       0.62      0.47      0.54       376

    accuracy                           0.78      1407
   macro avg       0.72      0.68      0.70      1407
weighted avg       0.77      0.78      0.77      1407



Interestingly, random forest here results in a strict improvement in Churn prediction - but doesn't cause an improvement significant enough to justify the feature engineering procedure for this dataset.

We conjecture that this is due to the binning of tenure and monthly charges discretising important data and making it easier to digest, but not being strong enough of an optimisation to result in substantial model feature learning differences. 
However, it can also be a signal for a mistake in EDA. As such this will be revisited in the future.